In [ ]:
import pandas as pd
import numpy as np
import optuna
import joblib
import os
import shutil
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from huggingface_hub import hf_hub_download

In [2]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

data_path = 'dataset_model/dataset_trainable_rev2.csv'

In [ ]:
# ==============================================================================
# 1. LOAD DATASET & FILTER KONTAMINASI PENGANGKUTAN SAMPAH (SELF-REVISION)
# ==============================================================================
if os.path.exists(data_path):
    print(f"Dataset Raw lokal sudah ada di '{data_path}'.")
else:
    print(f"Dataset Raw lokal belum ada. Mengunduh dataset dari Hugging Face...")
    
    temp_downloaded_path = hf_hub_download(
        repo_id="vel740/dhaka-smart-waste-processed",
        filename="processed/dataset_trainable_rev2.csv",
        repo_type="dataset"
    )
    
    os.makedirs('dataset_model', exist_ok=True)
    shutil.copy(temp_downloaded_path, data_path)
    print(f"Dataset selesai diunduh dan tersimpan di '{data_path}'.")

df = pd.read_csv(data_path)
print(f"Total baris awal: {len(df):,} baris")

In [ ]:

# Hitung selisih target dan lag ke-3 untuk mendeteksi event pickup (> 20% drop)
target_diff = df['target_next_hour'] - df['fill_level_percent']
velocity_3h = df['lag_2h'] - df['lag_3h']

# Hanya ambil fase akumulasi murni (bebas dari drop pengangkutan sampah)
clean_mask = (
    (target_diff > -20.0) & 
    (df['velocity_1h'] > -20.0) & 
    (df['velocity_2h'] > -20.0) & 
    (velocity_3h > -20.0)
)

df_clean = df[clean_mask].reset_index(drop=True)
print(f"Total baris setelah pembersihan event pickup: {len(df_clean):,} baris (Dibuang: {len(df) - len(df_clean)} baris)")

Total baris awal: 15,650 baris
Total baris setelah pembersihan event pickup: 15,650 baris (Dibuang: 0 baris)


In [4]:
df.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,is_weekend,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1735840800,67.709990,68.51879,66.02875,67.10285,42.061382,14.483168,0,-0.808800,2.490044,0.000000,1.000000,-0.433884,-0.900969,70.403190
1,0,1735844400,70.403190,67.70999,68.51879,66.02875,42.492905,15.378024,0,2.693199,-0.808800,0.258819,0.965926,-0.433884,-0.900969,69.584000
2,0,1735848000,69.584000,70.40319,67.70999,68.51879,44.316532,16.866007,0,-0.819191,2.693199,0.500000,0.866025,-0.433884,-0.900969,72.692300
3,0,1735851600,72.692300,69.58400,70.40319,67.70999,43.484684,18.462954,0,3.108299,-0.819191,0.707107,0.707107,-0.433884,-0.900969,68.450325
4,0,1735855200,68.450325,72.69230,69.58400,70.40319,44.515327,20.921236,0,-4.241974,3.108299,0.866025,0.500000,-0.433884,-0.900969,72.475070


In [5]:
df_clean.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,is_weekend,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1735840800,67.709990,68.51879,66.02875,67.10285,42.061382,14.483168,0,-0.808800,2.490044,0.000000,1.000000,-0.433884,-0.900969,70.403190
1,0,1735844400,70.403190,67.70999,68.51879,66.02875,42.492905,15.378024,0,2.693199,-0.808800,0.258819,0.965926,-0.433884,-0.900969,69.584000
2,0,1735848000,69.584000,70.40319,67.70999,68.51879,44.316532,16.866007,0,-0.819191,2.693199,0.500000,0.866025,-0.433884,-0.900969,72.692300
3,0,1735851600,72.692300,69.58400,70.40319,67.70999,43.484684,18.462954,0,3.108299,-0.819191,0.707107,0.707107,-0.433884,-0.900969,68.450325
4,0,1735855200,68.450325,72.69230,69.58400,70.40319,44.515327,20.921236,0,-4.241974,3.108299,0.866025,0.500000,-0.433884,-0.900969,72.475070


In [6]:
# ==============================================================================
# 2. DEFINISI FITUR & TRAIN-TEST SPLIT
# ==============================================================================
features = [
    'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h', 
    'lag_24h', 'lag_48h', 'is_weekend',
    'velocity_1h', 'velocity_2h', 
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]
target = 'target_next_hour'

# 1. Temporal Train-Test Split (Standar Time-Series)
split_timestamp = df_clean['timestamp_unix'].quantile(0.8)

train_mask = df_clean['timestamp_unix'] <= split_timestamp
test_mask = df_clean['timestamp_unix'] > split_timestamp

X_train = df_clean.loc[train_mask, features]
y_train = df_clean.loc[train_mask, target]
# Grouping tetap menggunakan bin_index untuk Optuna Cross-Validation
groups_train = df_clean.loc[train_mask, 'bin_index']

X_test = df_clean.loc[test_mask, features]
y_test = df_clean.loc[test_mask, target]

print(f"Batas waktu split: {pd.to_datetime(split_timestamp, unit='s')}")
print(f"Data Training: {len(X_train):,} baris")
print(f"Data Testing: {len(X_test):,} baris")

# Evaluasi Baseline LightGBM (Default) pada data bersih
base_lgbm = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
base_lgbm.fit(X_train, y_train)
pred_base = base_lgbm.predict(X_test)
print(f"\n[Baseline LightGBM - Data Bersih] MAE: {mean_absolute_error(y_test, pred_base):.4f}%")

Batas waktu split: 2025-01-13 21:00:00
Data Training: 12,549 baris
Data Testing: 3,101 baris

[Baseline LightGBM - Data Bersih] MAE: 1.6079%


In [7]:
# ==============================================================================
# 3. HYPERPARAMETER TUNING DENGAN OPTUNA (.SQLITE PERSISTENCE) + GROUP K-FOLD
# ==============================================================================
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 7, 127),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    gkf = GroupKFold(n_splits=5)
    cv_maes = []
    
    for tr_idx, val_idx in gkf.split(X_train, y_train, groups=groups_train):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        cv_maes.append(mean_absolute_error(y_va, preds))
        
    return np.mean(cv_maes)

# Konfigurasi SQLite Storage agar studi bisa di-pause dan dilanjutkan kapan saja
db_filename = "sqlite:///optuna_smartbin_study.sqlite3"
study_name = "lgbm_smartbin_16days_v3"

study = optuna.create_study(
    study_name=study_name,
    storage=db_filename,
    direction='minimize',
    load_if_exists=True  # Kunci utama agar otomatis resume jika file sudah ada
)

trials_existing = len(study.trials)
trials_to_add = 0

print(f"Status Database '{study_name}': Ditemukan {trials_existing} trials sebelumnya.")
print(f"Menjalankan tambahan {trials_to_add} trials baru...")

study.optimize(objective, n_trials=trials_to_add, show_progress_bar=True)

print("\n=== STATUS OPTUNA DATABASE ===")
print(f"Total akumulasi trials tersimpan : {len(study.trials)} trials")
print(f"Best Cross-Validation MAE        : {study.best_value:.4f}%")
print("Parameter Terbaik Saat Ini       :", study.best_params)

Status Database 'lgbm_smartbin_16days_v3': Ditemukan 300 trials sebelumnya.
Menjalankan tambahan 0 trials baru...

=== STATUS OPTUNA DATABASE ===
Total akumulasi trials tersimpan : 300 trials
Best Cross-Validation MAE        : 1.6717%
Parameter Terbaik Saat Ini       : {'n_estimators': 956, 'learning_rate': 0.016187306838693714, 'num_leaves': 20, 'max_depth': 10, 'min_child_samples': 8, 'subsample': 0.5045309145160505, 'subsample_freq': 1, 'colsample_bytree': 0.9970980981156482, 'reg_alpha': 0.009815734196381425, 'reg_lambda': 6.348627125971431e-08}


In [10]:
# ==============================================================================
# 4. EVALUASI & SIMPAN MODEL DARI DATABASE TERBAIK
# ==============================================================================
best_params = study.best_params.copy()
best_params.update({'random_state': 42, 'n_jobs': -1, 'verbose': -1})

final_model = LGBMRegressor(**best_params)
final_model.fit(X_train, y_train)

final_preds = final_model.predict(X_test)
final_mae = mean_absolute_error(y_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))
high_fill_mask = X_test['fill_level_percent'] >= 75.0
high_fill_mae = mean_absolute_error(y_test[high_fill_mask], final_preds[high_fill_mask])

print("\n=== HASIL EVALUASI MODEL FINAL (DARI SQLITE STUDY) ===")
print(f"Overall MAE (Semua Kondisi)      : {final_mae:.4f}%")
print(f"Overall RMSE                     : {final_rmse:.4f}%")
print(f"High-Fill MAE (Kapasitas >= 75%) : {high_fill_mae:.4f}%")

save_dir = 'model'
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, 'lightgbm_smartbin_16days_final.pkl')

joblib.dump(final_model, model_path)
print(f"\nModel berhasil disimpan dengan aman pada path: '{model_path}'")


=== HASIL EVALUASI MODEL FINAL (DARI SQLITE STUDY) ===
Overall MAE (Semua Kondisi)      : 1.5928%
Overall RMSE                     : 2.0123%
High-Fill MAE (Kapasitas >= 75%) : 1.6012%

Model berhasil disimpan dengan aman pada path: 'model\lightgbm_smartbin_16days_final.pkl'
